# ML Deployment — Assignment 2
### Model Saving, Loading, Versioning & Best Practices

---
## Question 1
**Train a simple scikit-learn DecisionTreeClassifier on a small dataset of your choice (for example, a list of songs with genres from your Spotify playlist), then use Pickle to save the trained model to a file named `song_genre_model.pkl`.**

In [ ]:
import os
import pickle
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# --- Create assignment2 folder for all generated files ---
os.makedirs('assignment2', exist_ok=True)

# --- Sample Spotify-like Song Dataset ---
# Features: [tempo (BPM), energy (0-100), danceability (0-100)]
# Labels:   Genre

X_train = np.array([
    [120, 80, 75],   # Pop
    [128, 90, 85],   # Pop
    [95,  40, 30],   # Classical
    [80,  30, 25],   # Classical
    [140, 95, 60],   # Rock
    [150, 92, 55],   # Rock
    [100, 70, 90],   # Hip-Hop
    [90,  75, 88],   # Hip-Hop
    [130, 85, 80],   # Pop
    [70,  25, 20],   # Classical
])

y_train = ['Pop', 'Pop', 'Classical', 'Classical', 'Rock', 'Rock',
           'Hip-Hop', 'Hip-Hop', 'Pop', 'Classical']

# --- Train the DecisionTreeClassifier ---
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Training Accuracy: {dt_model.score(X_train, y_train) * 100:.1f}%")

# --- Save the model using Pickle ---
with open('assignment2/song_genre_model.pkl', 'wb') as file:
    pickle.dump(dt_model, file)

print("Model saved as 'assignment2/song_genre_model.pkl'")

---
## Question 2
**Load the `song_genre_model.pkl` file you saved earlier and use it to predict the genre for a new song entry, printing the predicted result.**

In [ ]:
import pickle
import numpy as np

# --- Load the saved model from assignment2 folder ---
with open('assignment2/song_genre_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully from 'assignment2/song_genre_model.pkl'")

# --- New song entry for prediction ---
# Features: [tempo (BPM), energy (0-100), danceability (0-100)]
new_song = np.array([[110, 72, 82]])  # A song with moderate tempo, good energy & danceability

# --- Predict the genre ---
predicted_genre = loaded_model.predict(new_song)

print(f"\nNew Song Features -> Tempo: 110 BPM, Energy: 72, Danceability: 82")
print(f"Predicted Genre:  {predicted_genre[0]}")

---
## Question 3
**Train a scikit-learn KNeighborsClassifier to predict whether a Flipkart product is 'electronics' or 'fashion' based on sample features, then use Joblib to save the model as `product_category_model.joblib`.**

*Hint: Use `joblib.dump()` for saving and `joblib.load()` for loading.*

In [ ]:
import joblib
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# --- Sample Flipkart Product Dataset ---
# Features: [price (₹), weight (grams), avg_rating (1-5)]
# Labels:   Category ('electronics' or 'fashion')

X_train = np.array([
    [15000, 350, 4.2],   # electronics (phone)
    [45000, 2200, 4.5],  # electronics (laptop)
    [2500,  180, 4.0],   # electronics (earphones)
    [8000,  800, 4.3],   # electronics (tablet)
    [30000, 1500, 4.1],  # electronics (camera)
    [800,   200, 3.8],   # fashion (t-shirt)
    [1500,  300, 4.1],   # fashion (jeans)
    [2000,  150, 4.4],   # fashion (shoes)
    [500,   100, 3.9],   # fashion (scarf)
    [3000,  400, 4.2],   # fashion (jacket)
])

y_train = ['electronics', 'electronics', 'electronics', 'electronics', 'electronics',
           'fashion', 'fashion', 'fashion', 'fashion', 'fashion']

# --- Train the KNeighborsClassifier ---
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train, y_train)

print("KNN Model trained successfully!")
print(f"Training Accuracy: {knn_model.score(X_train, y_train) * 100:.1f}%")

# --- Save the model using Joblib ---
joblib.dump(knn_model, 'assignment2/product_category_model.joblib')

print("Model saved as 'assignment2/product_category_model.joblib'")

# --- Verify by loading and predicting ---
loaded_knn = joblib.load('assignment2/product_category_model.joblib')
new_product = np.array([[12000, 500, 4.0]])  # A new product
prediction = loaded_knn.predict(new_product)

print(f"\nNew Product -> Price: ₹12,000, Weight: 500g, Rating: 4.0")
print(f"Predicted Category: {prediction[0]}")

---
## Question 4
**Suppose you have trained and saved two versions of a model (v1 and v2) for predicting Zomato restaurant ratings. Create a folder structure for model versioning (e.g., `models/v1/`, `models/v2/`) and explain in a comment how you would keep track of which version is currently deployed.**

In [ ]:
import os
import json
import pickle
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from datetime import datetime

# ====================================================================
# MODEL VERSIONING STRATEGY (Comment)
# ====================================================================
#
# To keep track of which version is currently deployed, we use a
# 'model_registry.json' file at the root of the models/ folder.
#
# This registry stores:
#   - "current_version": the version currently in production
#   - "versions": a dict with metadata for each version including
#     training date, accuracy, description, and status.
#
# When deploying a new model:
#   1. Save it in a new versioned folder (e.g., models/v3/)
#   2. Update model_registry.json -> set "current_version" to "v3"
#   3. The serving application reads model_registry.json at startup
#      to know which model file to load.
#
# This approach allows easy rollback (just change current_version
# back to "v1"), A/B testing (serve both versions to different users),
# and full audit trail of all model versions.
# ====================================================================

# --- Create Folder Structure inside assignment2/ ---
os.makedirs('assignment2/models/v1', exist_ok=True)
os.makedirs('assignment2/models/v2', exist_ok=True)

# --- Sample Dataset: Zomato Restaurant Ratings ---
# Features: [avg_cost_for_two, num_reviews, delivery_time_min]
# Target:   rating (1.0 - 5.0)

X = np.array([
    [300, 150, 30],
    [800, 500, 25],
    [200, 50,  45],
    [1500, 1200, 20],
    [400, 300, 35],
    [600, 800, 28],
    [150, 30,  50],
    [1000, 900, 22],
])
y = np.array([3.5, 4.2, 2.8, 4.7, 3.8, 4.4, 2.5, 4.5])

# --- Train and Save Model v1 (max_depth=2, simpler model) ---
model_v1 = DecisionTreeRegressor(max_depth=2, random_state=42)
model_v1.fit(X, y)

with open('assignment2/models/v1/zomato_rating_model.pkl', 'wb') as f:
    pickle.dump(model_v1, f)

print(f"Model v1 saved | Accuracy (R²): {model_v1.score(X, y):.3f}")

# --- Train and Save Model v2 (max_depth=4, more complex model) ---
model_v2 = DecisionTreeRegressor(max_depth=4, random_state=42)
model_v2.fit(X, y)

with open('assignment2/models/v2/zomato_rating_model.pkl', 'wb') as f:
    pickle.dump(model_v2, f)

print(f"Model v2 saved | Accuracy (R²): {model_v2.score(X, y):.3f}")

# --- Create Model Registry ---
registry = {
    "current_version": "v2",
    "versions": {
        "v1": {
            "path": "assignment2/models/v1/zomato_rating_model.pkl",
            "trained_on": datetime.now().strftime("%Y-%m-%d"),
            "r2_score": round(model_v1.score(X, y), 3),
            "description": "Baseline model with max_depth=2",
            "status": "archived"
        },
        "v2": {
            "path": "assignment2/models/v2/zomato_rating_model.pkl",
            "trained_on": datetime.now().strftime("%Y-%m-%d"),
            "r2_score": round(model_v2.score(X, y), 3),
            "description": "Improved model with max_depth=4",
            "status": "deployed"
        }
    }
}

with open('assignment2/models/model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

print("\n--- Model Registry ---")
print(json.dumps(registry, indent=2))

# --- Show Folder Structure ---
print("\n--- Folder Structure ---")
for root, dirs, files in os.walk('assignment2'):
    level = root.replace('assignment2', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files:
        print(f"{subindent}📄 {file}")

---
## Question 5
**List three best practices for storing and managing machine learning model files in a real-world app like BookMyShow, and briefly explain why each is important.**

### Answer:

**1. Use Model Versioning with a Registry:**
Always store each trained model in a versioned folder (e.g., `models/v1/`, `models/v2/`) and maintain a model registry (a JSON file or a database) that records which version is currently deployed, along with metadata like training date, accuracy, and dataset used. This is important because it allows **instant rollback** — if a newly deployed model starts making bad recommendations on BookMyShow (e.g., suggesting horror movies to kids), the team can immediately switch back to the previous stable version without retraining.

**2. Store Models in Cloud Object Storage (e.g., AWS S3, Google Cloud Storage):**
Never store production model files only on a local machine or a single server. Use cloud storage with proper access controls and redundancy. This is important because model files can be large (hundreds of MBs), and if the server crashes or disk fails, the model is lost. Cloud storage provides **durability, availability, and easy access** from multiple deployment environments (staging, production, etc.).

**3. Log Training Metadata and Environment Details:**
Along with the model file, always save the exact Python/library versions (e.g., `scikit-learn==1.3.0`), training hyperparameters, training data snapshot or hash, and evaluation metrics. This is important because **model reproducibility** is critical — if a bug is discovered 6 months later, the team needs to know exactly how the model was trained to reproduce and fix the issue. It also prevents version mismatch errors where a model saved with one library version fails to load with another.